# Cox explanations with the complete training background

To make the high-dimensional survival experiments more representative, we replace the four-patient background with all training patients: **339 for GSE24080 and 383 for TCGA LGG methylation**. The background defines the empirical distribution used to average over missing features. Changing it changes the explanation target.

The saved model, preprocessing, split and three held-out patients are unchanged. The full training backgrounds avoid additional random subsampling; they do not guarantee representativeness of a future population. A separate sensitivity experiment found that even 100 randomly selected training patients could give substantially different explanations.

This notebook **runs** the new tolerance/timing sweep on the laptop GPU, using prepared inputs and independently validated float64 references. It does not rerun model fitting, the sensitivity study or reference construction; commands for those stages are in the generated report. Historical four-background results remain in `cox_gpu_tolerance`.

In [1]:
from pathlib import Path
import json
import sys
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]
from benchmarks import cox_background_experiment as experiment
from benchmarks.cox_background_report import build_report

_threads = threadpool_limits(limits=4)
started = perf_counter()
OUT = experiment.OUTPUT
info = experiment.environment(require_gpu=True)
print("GPU devices:", info["devices"])
print("Backend:", info["backend"], "JIT disabled:", info["jit_disabled"])


Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!


Metal device set to: Apple M4 Pro
GPU devices: ['METAL:0']
Backend: METAL JIT disabled: False


W0000 00:00:1789350067.797856  565678 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1789350067.834731  565678 service.cc:145] XLA service 0x8340e5100 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789350067.834750  565678 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1789350067.838411  565678 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1789350067.838423  565678 mps_client.cc:384] XLA backend will use up to 19069190144 bytes on device 0 for SimpleAllocator.


## Cohorts and independent reference

Both cohorts have hundreds of patients and tens or hundreds of thousands of features. Here `n_train` counts patients used to fit the Cox model; `d` counts the retained features. Every training patient is now included in the uniformly weighted background, with no test-patient leakage.

An independent double-precision implementation computes 64-node references. The theoretical quadrature bound is below $10^{-10}$ for all six cases. A 96-node crosscheck covers the first patient in each cohort. These are tightly bounded approximations, **not full exact-degree quadrature or exhaustive coalition enumeration**.

In [2]:
designs, checks = [], []
for name in experiment.DATASETS:
    design = json.loads((OUT / name / "design.json").read_text())
    check = json.loads((OUT / name / "reference_validation.json").read_text())
    assert max(check["quadrature_bounds"]) < 1e-10
    assert check["crosscheck_max_absolute_difference"] < 1e-6
    assert design["primary_background_size"] == design["n_train"]
    reference = np.load(OUT / name / "reference_full_64.npy", mmap_mode="r")
    assert reference.shape == (3, design["d"]) and np.isfinite(reference).all()
    designs.append({k: design[k] for k in ("dataset", "n_total", "n_train", "n_test", "d", "d_per_training_patient", "primary_background_size")})
    checks.append({"dataset": name, **check})
display(pd.DataFrame(designs))
display(pd.DataFrame(checks)[["dataset", "m_q", "crosscheck_max_absolute_difference", "reference_seconds", "crosscheck_seconds"]])


,dataset,n_total,n_train,n_test,d,d_per_training_patient,primary_background_size
0,gse24080,553,339,214,54675,161.283186,339
1,tcga_lgg_methylation,511,383,128,396065,1034.112272,383


,dataset,m_q,crosscheck_max_absolute_difference,reference_seconds,crosscheck_seconds
0,gse24080,64,2.501110e-12,22.535649,11.100853
1,tcga_lgg_methylation,64,1.409717e-11,181.854865,89.667760


## Background sensitivity

The following **saved** measurements compare different background selections using the same 128-node GPU rule. A difference here reflects a changed empirical explanation target, not quadrature error. Relative L2 difference is the length of the difference vector divided by the length of the full-background attribution vector. Top-20 overlap measures agreement among the largest-magnitude feature attributions.

In [3]:
sensitivity = pd.read_csv(OUT / "background_sensitivity.csv")
selected = sensitivity.patient.eq(0) & sensitivity.selection.isin(["historical_4", "nested_100", "nested_200", "full_training"])
display(sensitivity.loc[selected, ["dataset", "selection", "background_size", "mean_hazard_ratio_to_full", "relative_l2_error", "top20_overlap"]])


,dataset,selection,background_size,mean_hazard_ratio_to_full,relative_l2_error,top20_overlap
2,gse24080,nested_100,100,2.497854e+00,1.595585,0.80
3,gse24080,nested_200,200,1.250849e+00,0.361247,0.80
4,gse24080,full_training,339,1.000000e+00,0.000000,1.00
5,gse24080,historical_4,4,9.999137e-05,0.999598,0.25
29,tcga_lgg_methylation,nested_100,100,9.379387e-02,0.945025,0.10
30,tcga_lgg_methylation,nested_200,200,1.489876e+00,0.628454,0.90
31,tcga_lgg_methylation,full_training,383,1.000000e+00,0.000000,1.00
32,tcga_lgg_methylation,historical_4,4,7.563182e-09,0.999996,0.05


## Run the full-background GPU experiment

For each of three held-out patients and four absolute quadrature tolerances ($10^{-1}$ through $10^{-4}$), record the first API call and three warmed repetitions. The API returns a NumPy result, so the measured time includes GPU completion. It also includes node selection, factor preparation and transfers; it excludes loading, fitting and explainer construction. The CPU prefix implementation is measured for the first patient at $10^{-3}$ under the same background.

Metal evaluates in float32. The integration certificate does not bound floating-point error, and the original hazard scale is retained. We record whether the observed total absolute error actually meets each tolerance, rather than assuming that the certificate implies this. Full exact-degree timings from the old four-background experiment are not reused.

In [4]:
sweep_started = perf_counter()
experiment.main_sweep()
print(f"Measured sweep wall time: {perf_counter() - sweep_started:.2f} s")


gse24080 logspace_jax, patient 0, eps=0.1: 0.712s, 51 nodes, max error 0.00372


gse24080 logspace_jax, patient 1, eps=0.1: 0.700s, 50 nodes, max error 0.00417


gse24080 logspace_jax, patient 2, eps=0.1: 0.668s, 46 nodes, max error 0.00288


gse24080 logspace_jax, patient 0, eps=0.01: 0.704s, 52 nodes, max error 0.00192


gse24080 logspace_jax, patient 1, eps=0.01: 0.716s, 51 nodes, max error 0.00599


...

tcga_lgg_methylation logspace_jax, patient 0, eps=0.0001: 7.261s, 49 nodes, max error 0.0907


tcga_lgg_methylation logspace_jax, patient 1, eps=0.0001: 7.354s, 49 nodes, max error 0.216


tcga_lgg_methylation logspace_jax, patient 2, eps=0.0001: 7.127s, 48 nodes, max error 0.0693


tcga_lgg_methylation prefix_scan_numpy, patient 0, eps=0.001: 49.140s, 48 nodes, max error 3.48e-10


Measured sweep wall time: 616.66 s


In [5]:
timings = pd.read_csv(OUT / "full_background_timings.csv")
assert len(timings) == 26
assert len(pd.read_csv(OUT / "raw_timings.csv")) == 104
columns = ["dataset", "backend", "patient", "eps", "m_q", "median_seconds", "median_budget_seconds", "max_absolute_error", "relative_l2_error", "observed_absolute_tolerance_met"]
display(timings.loc[timings.patient.eq(0), columns])
print("Report:", build_report())


,dataset,backend,patient,eps,m_q,median_seconds,median_budget_seconds,max_absolute_error,relative_l2_error,observed_absolute_tolerance_met
0,gse24080,logspace_jax,0,0.1000,51,0.711734,0.229804,3.715481e-03,4.331233e-06,True
3,gse24080,logspace_jax,0,0.0100,52,0.703603,0.221368,1.924277e-03,2.463905e-06,True
6,gse24080,logspace_jax,0,0.0010,53,0.707555,0.220087,2.550312e-03,2.961578e-06,False
9,gse24080,logspace_jax,0,0.0001,54,0.717456,0.233521,3.307017e-03,3.834260e-06,False
12,gse24080,prefix_scan_numpy,0,0.0010,53,6.790522,0.291309,4.217782e-11,5.505296e-14,True
13,tcga_lgg_methylation,logspace_jax,0,0.1000,46,7.303784,2.001728,6.562135e-02,9.537087e-06,True
16,tcga_lgg_methylation,logspace_jax,0,0.0100,47,7.219925,1.949470,6.710907e-02,1.001028e-05,False
19,tcga_lgg_methylation,logspace_jax,0,0.0010,48,7.795597,2.508162,1.057670e-01,1.647450e-05,False
22,tcga_lgg_methylation,logspace_jax,0,0.0001,49,7.261389,1.945891,9.066572e-02,1.385217e-05,False
25,tcga_lgg_methylation,prefix_scan_numpy,0,0.0010,48,49.139777,1.875257,3.478817e-10,1.738359e-13,True


Report: /Users/siulunchau/Library/Mobile Documents/com~apple~CloudDocs/Documents/research_projects/current/QuadraSHAP/benchmarks/results/cox_background_sensitivity/README.md


## Updated manuscript paragraph

The paragraph uses measured full-background times. Exact node counts are an algebraic comparison; no new full exact-degree timing was collected for these backgrounds. See the report's accuracy table for any finite-precision tolerance failures.

The manuscript paragraph below was revised after the completed benchmark to include quadrature-rule setup time. This editorial update did not rerun the experiment. The original notebook execution took 618.37 seconds.

```latex
To illustrate QuadraSHAP in the small-$n$, extremely large-$d$ regime, we explain Cox models for GSE24080 gene expression and TCGA lower-grade glioma methylation, with $(n_{\mathrm{train}},d)=(339,54{,}675)$ and $(383,396{,}065)$, respectively. All training patients serve as uniformly weighted backgrounds. For the first held-out patient in each cohort, a quadrature tolerance of $10^{-3}$ selects only $53$ and $48$ nodes, compared with the $27{,}338$ and $198{,}033$ Gauss--Legendre nodes sufficient for exact quadrature. Median warm explanation times on an Apple M4 Pro GPU are $0.71$ and $7.80$ seconds, including automatic node selection (Table~\ref{tab:cox-background}). Constructing very large quadrature rules can itself impose substantial startup latency: generating the $198{,}033$-node rule took approximately $562$ seconds in our earlier benchmark. Using substantially fewer nodes therefore also reduces setup costs when a precomputed rule is unavailable. The quadrature certificate excludes floating-point error; on these fits, single-precision GPU evaluation exceeds the requested absolute tolerance, as reported separately in the table.
```